# 07 CLASSIFY: classification at four target windows

Logistic Regression, XGBoost and CatBoost on the 17 measurement channels of the June 2025 telemetry, under five class-balancing strategies (default class weighting, undersampling, random oversampling, SMOTE and ADASYN), at the detection window D5 and the warning windows W5, W60 and W300.

* **Partition.** Chronological: the final fifth of the record is the test partition and the final fifth of the remainder a validation block; boundaries are moved so that no run of positive seconds is split.
* **Fitting.** Boosted models are fitted on the inner training block with early stopping on validation PR-AUC (patience 100), then refitted on the whole training partition at the iteration count found; the test partition is scored once.
* **Class weighting.** "Default (class weight)" weights all three models (for XGBoost, `scale_pos_weight` = negatives / positives).
* **Operating points.** Precision is read off the test precision-recall curve at recall 0.25 and 0.50.

This notebook gives seed 42; Tables 6 to 8 report means over five seeds from `08_SEEDS`. Input `build/telemetry_labelled_v4.pkl`; outputs in `OUTPUT/v4/` (the output file names carry an internal suffix, `_v3`).

Paper: Sections 2.7 and 3.2; Tables 4 and 9; Figures 5 and 6.

In [ ]:
# =====================================================================
# 07_CLASSIFY: configuration
# =====================================================================
import os, warnings, numpy as np, pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

CFG = dict(
    labelled      = "build/telemetry_labelled_v4.pkl",
    outdir        = "OUTPUT/v4",
    horizons      = ["D5", "W5", "W60", "W300"],
    detection     = ["D5"],
    horizon_label = {"D5": "±5 s (detection)", "W5": "5 s", "W60": "1 min", "W300": "5 min"},
    expected_run  = {"D5": 11, "W5": 5, "W60": 60, "W300": 300},
    patience      = 100,     # early-stopping rounds (50 in CLASSIFY2)
    test_frac     = 0.20,   # final 20% of the record, scored once
    val_frac      = 0.20,   # final 20% of the training partition, early stopping only
    seed          = 42,
    roll_windows  = [60],
    sample_frac   = 1.0,    # < 1.0 for a fast development pass
    exclude_state = True,
    use_rolling   = False,
    exclude_op    = True,
    recall_points = [0.25, 0.50],
)

STATE_CHANNELS = ["Traction_Interlock", "Brk_Ele_Hlth", "Isolating_Emg_Exhaust", "Vig_Warning", "Vig_Emg_Brk", "Emergency_Brake_Applied", "Panto_Status", "Panto_1_Pr_Sw", "Panto_2_Pr_Sw", "VCB_Status", "Motoring", "DB", "BC1_Pr_Sw", "BC2_Pr_Sw"]
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE", "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4", "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
BINARY = STATE_CHANNELS + WHEEL_SLIP          # 14 + 5 = 19
ROLL_CHANNELS = ["KMH", "AMP", "KV", "BG1_DCLV", "BG2_DCLV", "PRESSURE", "TRACTION_PCT"]
OP_MODES = ["POWERING", "PROPULSION", "REGENERATIVE BRAKING", "VCB ON", "IDLE"]
os.makedirs(CFG["outdir"], exist_ok=True)

## 0. Label diagnostics

For each target: positive, negative and excluded (−1) rows, and the length of
runs of consecutive positive seconds. For an isolated onset the run equals the
window: 11 for D5, 5 for W5, 60 for W60 and 300 for W300. Warning runs can be
shorter where an earlier episode is still active.

In [ ]:
lab = (pd.read_pickle(CFG["labelled"])[["DATETIME"] + CFG["horizons"]].sort_values("DATETIME").reset_index(drop=True))
gap1 = lab["DATETIME"].diff().dt.total_seconds().eq(1)

def runs_of(mask):
    """Lengths of runs where mask is True on consecutive one-second rows,
    plus the label of the row that follows each run."""
    start = mask & ~(mask.shift(fill_value=False) & gap1)
    rid = start.cumsum().where(mask)
    g = lab.index.to_series()[mask].groupby(rid[mask])
    first, last = g.min(), g.max()
    return (last - first + 1).values, last.values

diag = []
for h in CFG["horizons"]:
    y = lab[h]
    n_pos, n_neg, n_exc = int((y == 1).sum()), int((y == 0).sum()), int((y == -1).sum())
    L, last = runs_of(y == 1)
    nxt = lab[h].reindex(last + 1)
    vals, cnts = np.unique(L, return_counts=True)
    mode = int(vals[cnts.argmax()])
    Lx, _ = runs_of(y == -1)
    diag.append(dict(
        horizon=h, positive=n_pos, negative=n_neg, excluded=n_exc,
        pos_share_pct=round(100 * n_pos / (n_pos + n_neg), 3),
        pos_runs=len(L), run_mode=mode,
        share_runs_at_mode_pct=round(100 * cnts.max() / len(L), 1),
        run_median=float(np.median(L)), run_p90=float(np.percentile(L, 90)),
        run_max=int(L.max()),
        runs_followed_by_excluded_pct=round(100 * (nxt == -1).mean(), 1),
        excluded_runs=len(Lx), excluded_run_median=float(np.median(Lx)),
    ))
    print(f"--- {h}: modal positive run {mode} rows (expected {CFG['expected_run'][h]}) "
          f"({cnts.max():,} of {len(L):,} runs); "
          f"lengths 1-10: {dict(zip(vals[vals <= 10].tolist(), cnts[vals <= 10].tolist()))}")

diag = pd.DataFrame(diag).set_index("horizon")
print("\n", diag.T.to_string())
diag.to_csv(f"{CFG['outdir']}/label_diagnostics_v3.csv")
del lab

## 1. Table 4: measurement channels used in the models
Statistics over the June 2025 study month, after the validity specification. `valid` counts non-missing values.

In [ ]:
DESCR = {
    "KMH":          "Speed (km/h)",
    "DIST_M":       "Distance per record (m)",
    "KV":           "Catenary voltage (kV)",
    "AMP":          "Catenary current (A)",
    "KWH":          "Energy per record (kWh)",
    "BV":           "Battery voltage (V)",
    "PRESSURE":     "Brake pipe pressure (kg/cm2)",
    "TRACTION_PCT": "Traction effort demand (%)",
    "BG1_KN":       "Bogie 1 tractive/braking force (kN)",
    "BG2_KN":       "Bogie 2 tractive/braking force (kN)",
    "BG1_DCLV":     "Bogie 1 DC link voltage (V)",
    "BG2_DCLV":     "Bogie 2 DC link voltage (V)",
}

tel = pd.read_pickle(CFG["labelled"])
t3 = pd.DataFrame({
    c: dict(description=DESCR[c], valid=int(tel[c].notna().sum()),
            mean=tel[c].mean(), sd=tel[c].std(),
            min=tel[c].min(), p01=tel[c].quantile(0.01),
            p99=tel[c].quantile(0.99), max=tel[c].max())
    for c in NUMERIC}).T
print(f"{len(tel):,} rows\n")
print(t3.to_string(float_format=lambda v: f"{v:,.2f}"))
t3.to_csv(f"{CFG['outdir']}/table3_numeric_v3.csv")

slip = pd.DataFrame({c: dict(seconds_asserted=int((tel[c] == 1).sum()),
                             share_pct=100 * (tel[c] == 1).mean())
                     for c in WHEEL_SLIP}).T
print("\nWheel-slip indicators in the primary specification")
print(slip.to_string(float_format=lambda v: f"{v:.4f}"))
slip.to_csv(f"{CFG['outdir']}/wheel_slip_prevalence_v3.csv")
del tel

## 2. Functions

In [ ]:
def add_features(df, cfg):
    df = df.sort_values("DATETIME").reset_index(drop=True)
    df["OP"] = df["OP"].astype(str)
    for m in OP_MODES:
        df["OP_" + m.replace(" ", "_")] = (df["OP"] == m).astype("int8")
    gap = df["DATETIME"].diff().dt.total_seconds()
    for w in cfg["roll_windows"]:
        contiguous = gap.rolling(w, min_periods=w).max() == 1
        for c in [c for c in ROLL_CHANNELS if c in df.columns]:
            r = df[c].rolling(w, min_periods=max(3, w // 2))
            df[f"{c}_m{w}"] = r.mean().where(contiguous)
            df[f"{c}_s{w}"] = r.std().where(contiguous)
            df[f"{c}_d{w}"] = (df[c] - df[c].shift(w)).where(contiguous)
    return df

def feature_columns(df, cfg):
    cols = [c for c in NUMERIC + BINARY if c in df.columns]
    cols += [c for c in df.columns if c.startswith("OP_")]
    if cfg.get("use_rolling", False):
        for w in cfg["roll_windows"]:
            cols += [c for c in df.columns if c.endswith((f"_m{w}", f"_s{w}", f"_d{w}"))]
    if cfg.get("exclude_state", False):
        cols = [c for c in cols if c not in STATE_CHANNELS]
    if cfg.get("exclude_op", False):
        cols = [c for c in cols if not c.startswith("OP_")]
    return cols

def chrono_cut(y, frac):
    """Index at which the final `frac` of y begins, moved forward so that
    no run of positive rows is divided."""
    n = len(y)
    i = int(n * (1 - frac))
    while i < n - 1 and y[i] == 1 and y[i - 1] == 1:
        i += 1
    return i

def three_way_split(y, cfg):
    """[0, v) inner training | [v, t) validation | [t, n) test.
    Training partition = [0, t). The test block matches CLASSIFY exactly."""
    n = len(y)
    t = int(n * (1 - cfg["test_frac"]))
    while t < n - 1 and y[t] == 1:        # same rule as CLASSIFY.chrono_split
        t += 1
    v = chrono_cut(y[:t], cfg["val_frac"])
    return np.arange(v), np.arange(v, t), np.arange(t), np.arange(t, n)

SAMPLERS = {
    "Default (class weight)": None,
    "Undersampling":          lambda s: RandomUnderSampler(random_state=s),
    "Random Oversampling":    lambda s: RandomOverSampler(random_state=s),
    "SMOTE":                  lambda s: SMOTE(random_state=s),
    "ADASYN":                 lambda s: ADASYN(random_state=s),
}
MODEL_NAMES = ["Logistic Regression", "XGBoost", "CatBoost"]

def make_model(name, seed, balanced, y, n_iter=None):
    """n_iter=None: search stage with early stopping on validation.
    n_iter=k: refit stage, fixed at the iteration count found on validation."""
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, random_state=seed,
                                  class_weight="balanced" if balanced else None)
    if name == "XGBoost":
        spw = float((y == 0).sum() / max((y == 1).sum(), 1)) if balanced else 1.0
        kw = dict(n_estimators=600, early_stopping_rounds=CFG["patience"]) if n_iter is None \
             else dict(n_estimators=n_iter)
        return XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8,
                             colsample_bytree=0.8, eval_metric="aucpr",
                             tree_method="hist", scale_pos_weight=spw,
                             random_state=seed, n_jobs=-1, **kw)
    if name == "CatBoost":
        kw = dict(iterations=800, early_stopping_rounds=CFG["patience"]) if n_iter is None \
             else dict(iterations=n_iter)
        return CatBoostClassifier(learning_rate=0.05, depth=6, eval_metric="PRAUC",
                                  auto_class_weights="Balanced" if balanced else None,
                                  random_seed=seed, verbose=0, **kw)

def resample(sampler, X, y, seed):
    return (X, y) if sampler is None else SAMPLERS[sampler](seed).fit_resample(X, y)

def fit_two_stage(name, sampler, Xa, ya, Xv, yv, Xtr, ytr, seed):
    """Stage 1: fit on inner training data, early-stop on validation.
    Stage 2: refit on the full training partition at that iteration count.
    Logistic Regression has no iteration budget and is fitted once, on the
    full training partition. Returns (model, scaler, n_iter)."""
    balanced = sampler is None
    if name == "Logistic Regression":
        sc = StandardScaler().fit(Xtr)
        Xs, ys = resample(sampler, sc.transform(Xtr), ytr, seed)
        m = make_model(name, seed, balanced, ys).fit(Xs, ys)
        return m, sc, None

    Xs, ys = resample(sampler, Xa, ya, seed)
    m1 = make_model(name, seed, balanced, ys)
    if name == "XGBoost":
        m1.fit(Xs, ys, eval_set=[(Xv, yv)], verbose=False)
        n_iter = int(m1.best_iteration) + 1
    else:
        m1.fit(Xs, ys, eval_set=(Xv, yv), use_best_model=True)
        n_iter = int(m1.get_best_iteration()) + 1

    Xs, ys = resample(sampler, Xtr, ytr, seed)
    m2 = make_model(name, seed, balanced, ys, n_iter=n_iter).fit(Xs, ys)
    return m2, None, n_iter

def score(m, sc, X):
    return m.predict_proba(sc.transform(X) if sc is not None else X)[:, 1]

def evaluate(y, s, recall_points):
    p, r, _ = precision_recall_curve(y, s)
    out = dict(pr_auc=average_precision_score(y, s),
               roc_auc=roc_auc_score(y, s) if len(np.unique(y)) > 1 else np.nan)
    for rp in recall_points:
        out[f"precision_at_recall_{rp:.2f}"] = float(p[r >= rp].max())
    return out, (p, r)

## 3. Primary run

In [ ]:
def run(cfg):
    df = pd.read_pickle(cfg["labelled"])
    if cfg["sample_frac"] < 1.0:
        df = df.iloc[:int(len(df) * cfg["sample_frac"])].copy()
    df = add_features(df, cfg)
    cols = feature_columns(df, cfg)
    print(f"{len(df):,} rows, {len(cols)} features: {cols}\n")

    bar = tqdm(total=len(cfg["horizons"]) * len(SAMPLERS) * len(MODEL_NAMES), desc="fits")
    rows, curves = [], {}
    for h in cfg["horizons"]:
        d = df[df[h] != -1]
        X = d[cols].astype("float32").fillna(0).values
        y = d[h].astype(int).values
        ia, iv, itr, ite = three_way_split(y, cfg)
        base = y[ite].mean()
        tqdm.write(f"--- {h}: inner {len(ia):,} ({y[ia].mean()*100:.3f}% pos)  "
                   f"val {len(iv):,} ({y[iv].mean()*100:.3f}% pos)  "
                   f"test {len(ite):,} ({base*100:.3f}% pos)")
        for sname in SAMPLERS:
            for mname in MODEL_NAMES:
                try:
                    m, sc, n_iter = fit_two_stage(
                        mname, None if sname.startswith("Default") else sname,
                        X[ia], y[ia], X[iv], y[iv], X[itr], y[itr], cfg["seed"])
                    s = score(m, sc, X[ite])
                except Exception as e:
                    tqdm.write(f"    {sname} / {mname}: FAILED {e}")
                    bar.update(1); continue
                r, curve = evaluate(y[ite], s, cfg["recall_points"])
                r.update(horizon=h, sampling=sname, model=mname, n_iter=n_iter,
                         baseline_pr_auc=base, lift=r["pr_auc"] / base)
                rows.append(r); curves[(h, sname, mname)] = curve
                bar.update(1)
                tqdm.write(f"    {sname:24s} {mname:20s} PR-AUC {r['pr_auc']:.3f}  "
                           f"lift {r['lift']:.1f}x  iters {n_iter}")
        tqdm.write("")
    bar.close()
    res = pd.DataFrame(rows)
    lead = ["horizon", "sampling", "model", "pr_auc", "baseline_pr_auc", "lift", "roc_auc"]
    res = res[lead + [c for c in res.columns if c not in lead]]
    res.to_csv(f"{cfg['outdir']}/classification_results_v3.csv", index=False)
    return res, curves

res, curves = run(CFG)

## 4. Summary tables (seed 42; Tables 6 to 8 report five-seed means)

In [ ]:
for h in CFG["horizons"]:
    t = res[res.horizon == h].pivot(index="sampling", columns="model", values=["pr_auc", "lift"])
    t = t.reindex(list(SAMPLERS))
    print(f"--- {h}  ({CFG['horizon_label'][h]})   baseline "
          f"{res[res.horizon == h].baseline_pr_auc.iloc[0]:.4f}")
    print(t.round(3).to_string(), "\n")

best = (res.sort_values("pr_auc", ascending=False)
          .groupby(["horizon", "model"]).first().reset_index())
piv = best.pivot(index="model", columns="horizon", values="pr_auc").round(3)[CFG["horizons"]]
print("Best PR-AUC by model and horizon"); print(piv.to_string())
best.to_csv(f"{CFG['outdir']}/summary_best_by_horizon_v3.csv", index=False)

## 5. Figure 6: PR-AUC and lift by target window

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.7, "figure.dpi": 110,
})
PALETTE = {"XGBoost": "#1f6fb4", "CatBoost": "#c0392b", "Logistic Regression": "#e08a1e"}
MARKER  = {"XGBoost": "o", "CatBoost": "s", "Logistic Regression": "^"}
H  = CFG["horizons"]
HL = [CFG["horizon_label"][h].replace(" (detection)", "\n(detection)") for h in H]
DET = [i for i, h in enumerate(H) if h in CFG["detection"]]
WRN = [i for i, h in enumerate(H) if h not in CFG["detection"]]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for m, g in res.groupby("model"):
    for a, col in zip(ax, ["pr_auc", "lift"]):
        v = g.groupby("horizon")[col].max().reindex(H).values
        a.plot(WRN, v[WRN], marker=MARKER[m], ms=4, lw=1.7, color=PALETTE[m], label=m)
        a.plot(DET, v[DET], marker=MARKER[m], ms=4, lw=0, color=PALETTE[m])
for a, lab in zip(ax, ["PR-AUC", "Lift over baseline"]):
    a.axvline(0.5, color="0.6", lw=0.6, ls=(0, (2, 2)))
    a.set_xticks(range(len(H))); a.set_xticklabels(HL)
    a.set_ylabel(lab); a.set_xlabel("Target window")
    a.set_ylim(0, None); a.margins(x=0.08)
    a.grid(axis="y", color="0.85", lw=0.5)
    a.set_title(lab, loc="left", color="0.3")
    a.legend(frameon=False, handlelength=1.8, borderaxespad=0.2)
plt.tight_layout()
plt.savefig(f"{CFG['outdir']}/horizon_decay_v3.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{CFG['outdir']}/horizon_decay_v3.pdf", bbox_inches="tight")
plt.show()

## 6. Figure 5: precision-recall curves

In [ ]:
RANK_COLOR = ["#1f6fb4", "#c0392b", "#e08a1e"]
fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True)
for j, (a, h) in enumerate(zip(axes.ravel(), H)):
    b = res[res.horizon == h].sort_values("pr_auc", ascending=False).head(3)
    for i, (_, r) in enumerate(b.iterrows()):
        p, rc = curves[(h, r.sampling, r.model)]
        a.plot(rc, p, lw=1.6, color=RANK_COLOR[i],
               label=f"{r.model} / {r.sampling}  ({r.pr_auc:.3f})")
    a.axhline(b.baseline_pr_auc.iloc[0], ls=(0, (4, 3)), c="0.45", lw=0.9,
              label=f"Baseline ({b.baseline_pr_auc.iloc[0]:.4f})")
    a.set_xlim(0, 1); a.set_ylim(0, None)
    a.grid(color="0.85", lw=0.5)
    a.set_title(CFG["horizon_label"][h], loc="left", color="0.3")
    if j >= 2: a.set_xlabel("Recall")
    if j % 2 == 0: a.set_ylabel("Precision")
    a.legend(frameon=False, fontsize=8, handlelength=1.8, borderaxespad=0.2, labelspacing=0.3)
plt.tight_layout()
plt.savefig(f"{CFG['outdir']}/pr_curves_v3.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{CFG['outdir']}/pr_curves_v3.pdf", bbox_inches="tight")
plt.show()

## 7. Feature-set ablation (Table 9)

XGBoost with class weighting and the same two-stage protocol as Section 3, so the primary row equals the XGBoost "Default (class weight)" values above. The first row removes the DC link voltages and wheel-slip indicators from the primary set; the operating-mode row tests whether operating-mode flags add to the protection-state channels.

In [ ]:
_df = add_features(pd.read_pickle(CFG["labelled"]), CFG)
if CFG["sample_frac"] < 1.0:
    _df = _df.iloc[:int(len(_df) * CFG["sample_frac"])].copy()

PRIMARY = feature_columns(_df, CFG)
variants = {
    "Measurements, without DC link voltage and wheel slip":
        [c for c in PRIMARY if c not in ["BG1_DCLV", "BG2_DCLV"] + WHEEL_SLIP],
    "Measurements only (primary)": PRIMARY,
    "+ protection state":
        feature_columns(_df, {**CFG, "exclude_state": False}),
    "+ protection state + operating mode":
        feature_columns(_df, {**CFG, "exclude_state": False, "exclude_op": False}),
    "+ protection state + rolling":
        feature_columns(_df, {**CFG, "exclude_state": False, "use_rolling": True}),
}

tab, imps = {}, {}
for name, cols in tqdm(variants.items(), desc="ablation"):
    row = {"channels": len(cols)}
    for h in CFG["horizons"]:
        d = _df[_df[h] != -1]
        X = d[cols].astype("float32").fillna(0).values
        y = d[h].astype(int).values
        ia, iv, itr, ite = three_way_split(y, CFG)
        m, sc, _ = fit_two_stage("XGBoost", None, X[ia], y[ia], X[iv], y[iv],
                                 X[itr], y[itr], CFG["seed"])
        row[h] = round(average_precision_score(y[ite], score(m, sc, X[ite])), 3)
        imps[(name, h)] = pd.Series(m.feature_importances_, index=cols)
    tab[name] = row
    tqdm.write(f"  {name:55s} {row}")

abl = pd.DataFrame(tab).T[["channels"] + CFG["horizons"]]
prim = abl.loc["Measurements only (primary)", CFG["horizons"]]
for h in CFG["horizons"]:
    abl[f"change_{h}_pct"] = (100 * (abl[h] / prim[h] - 1)).round(1)
print("\n", abl.to_string())
abl.to_csv(f"{CFG['outdir']}/ablation_v3.csv")

chk = res[(res.model == "XGBoost") & (res.sampling == "Default (class weight)")] \
        .set_index("horizon").pr_auc.round(3)
print("\nConsistency check, primary row vs Section 3:",
      "OK" if all(abs(chk[h] - prim[h]) < 1e-9 for h in CFG["horizons"]) else "MISMATCH",
      dict(chk), dict(prim))

## 8. Channel importance, primary specification

In [ ]:
for h in CFG["horizons"]:
    print(f"--- {h} ({CFG['horizon_label'][h]})")
    print(imps[("Measurements only (primary)", h)].nlargest(12).round(4).to_string(), "\n")

___
**END**